# 🌌 HollyShit Phase 0 v3: 去空腔各向同性五师流形蒸馏 (`SensoryFuser + GodEncoder`)

- **统一多数据源管道 (ModelScope 默认 / Google Drive 暂存离线 / Hugging Face CDN / 本地目录)**:
  - **`ms` (默认)**: 零配置直连 ModelScope (`catclever/emb_npy`) 按需流式拉取，国内与境外全免翻极速畅连；
  - **`gdrive` (云盘持久离线源)**: 从 Google Drive 目录读取持久 `.npz` 分块，后台打工仔按需将其暂存 (Stage) 至本地高速 NVMe SSD (`/content/embs_cache`)，训练在 NVMe 进行，消费完毕自动回收本地暂存，对 Google Drive 云盘文件 100% 零修改、零误删、极速训练；
  - **`hf` (可选 CDN 流式)**: Hugging Face Hub 按需流式串流 (`drunkencat/emb_npy`)；
  - **`local` (本地磁盘)**: 直接读取本地已解压目录。
- **双重防丢权硬门禁 (`Mount Guard + HF Off-Drive Backup`)**: 每个关键单元格与训练主程序第 0 秒强制校验 `os.path.ismount('/content/drive')` 及软链接真实挂载表，严禁写入容器临时 SSD；同时支持自动将每次保存的 Checkpoint 异地备份至私有仓库 `<hf_user>/hollyshit-checkpoints`。
- **本地 NVMe SSD 零泄漏单块控流**: 分块直写 `/content/embs_cache`，配合 `--no-prefetch` 单块内存驻留（峰值内存约 `~10.2 GB`，消费完连同 `.metadata`/软链接源彻底清除）与 `--keep_last_ckpts 3` 滚动清理。
- **推荐 Colab 运行时规格**: **L4 / A100 GPU**（原生 `bfloat16` 零溢出），或 **T4 GPU + 务必开启「高内存 (High-RAM >= 25GB)」**。

In [ ]:
# 0. [可选] 清理旧版 Phase 0 v3 Checkpoint (仅当确认 Google Drive 已挂载且需从零重训时取消注释；默认保留以支持 --auto_resume 断点续训)
# import os; assert os.path.ismount('/content/drive'), '❌ /content/drive 未挂载！'
# !rm -rf /content/drive/MyDrive/hollyshit/checkpoints/p0_v3/p0_v3_5models_*


In [ ]:
import os
import shutil
from google.colab import drive

# 1. 挂载 Google Drive 并立即执行硬门禁校验 (防止切换运行时后未挂载写进临时盘!)
drive.mount('/content/drive', force_remount=False)
assert os.path.ismount('/content/drive'), '❌ Google Drive 未挂载成功！请授权后再运行后续单元格！'

# 2. 安装 Hugging Face 高速流式下载 (hf_transfer)、魔搭备用源与权重序列化依赖
!pip install -q "huggingface_hub[hf_transfer]" modelscope safetensors pyarrow || pip install -q huggingface_hub modelscope safetensors pyarrow

try:
    import hf_transfer  # noqa: F401
    os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
    print('⚡ 已启用 hf_transfer Rust 高速 CDN 多连接加速 (HF_HUB_ENABLE_HF_TRANSFER=1)')
except ImportError:
    os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
    print('ℹ️ 未检测到 hf_transfer 轮子，已安全使用标准 huggingface_hub 下载器')

# 3. 读取 Colab Secrets 中的 HF_TOKEN (及可选 MODELSCOPE_API_TOKEN)，自动激活数据集与私有权重双活备份仓
os.environ.setdefault('HF_CKPT_REPO', '')
try:
    from google.colab import userdata
    hf_tok = userdata.get('HF_TOKEN')
    if hf_tok and hf_tok.strip():
        hf_tok = hf_tok.strip()
        os.environ['HF_TOKEN'] = hf_tok
        from huggingface_hub import HfApi
        hf_user = HfApi(token=hf_tok).whoami()['name']
        os.environ['HF_DATASET_REPO'] = f'{hf_user}/emb_npy'
        os.environ['HF_CKPT_REPO'] = f'{hf_user}/hollyshit-checkpoints'
        print(f'🔑 已加载 HF_TOKEN (用户: {hf_user})')
        print(f'   • 优先 HF 数据集源 : {os.environ["HF_DATASET_REPO"]}')
        print(f'   • 异地私有权重备份 : {os.environ["HF_CKPT_REPO"]}')
except Exception as _hf_err:
    print(f'ℹ️ 未配置或无法解析 HF_TOKEN (将探测公开 HF 源或魔搭源，跳过 HF 私有权重镜像): {_hf_err}')

try:
    from google.colab import userdata
    ms_tok = userdata.get('MODELSCOPE_API_TOKEN')
    if ms_tok and ms_tok.strip():
        os.environ['MODELSCOPE_API_TOKEN'] = ms_tok.strip()
        print('🔑 已从 Colab Secrets 加载 MODELSCOPE_API_TOKEN')
except Exception:
    pass

# 4. 硬件规格与高内存 (High-RAM) 安全自检
import torch

def _get_system_ram_gb() -> float:
    if os.path.exists('/proc/meminfo'):
        with open('/proc/meminfo', 'r', encoding='utf-8') as f:
            for line in f:
                if line.startswith('MemTotal:'):
                    kb = int(line.split()[1])
                    return kb / (1024.0 ** 2)
    return 0.0

ram_gb = _get_system_ram_gb()
disk_free_gb = shutil.disk_usage('/content').free / (1024.0 ** 3)
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (未检测到 GPU!)'
vram_gb = (torch.cuda.get_device_properties(0).total_memory / (1024.0 ** 3)) if torch.cuda.is_available() else 0.0

print('\n' + '=' * 62)
print(f'🔥 GPU 设备   : {gpu_name} ({vram_gb:.1f} GB VRAM)')
print(f'🧠 系统内存   : {ram_gb:.1f} GB RAM')
print(f'💾 本地 SSD   : /content 可用空间 {disk_free_gb:.1f} GB')
print(f'📂 云盘挂载   : /content/drive ismount={os.path.ismount("/content/drive")}')
print('=' * 62)
if ram_gb > 0 and ram_gb < 18.0:
    print('⚠️ [重要警告] 当前系统内存 < 18 GB！单块 50w 五师特征解压约占 10.2 GB RAM。')
    print('   请在菜单栏「代码执行程序 -> 更改运行时类型」中开启【高内存 (High-RAM)】后再启动训练！')


In [ ]:
import os
if not os.path.ismount('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')
assert os.path.ismount('/content/drive'), '❌ /content/drive 未挂载！严禁在本地容器盘创建伪目录！'

# 从 GitHub 拉取纯代码并初始化 training/core 子模块 (如已存在则 git pull)
if not os.path.exists('/content/HollyShit'):
    !git clone https://github.com/catclever/HollyShit.git /content/HollyShit
    %cd /content/HollyShit
    !git submodule update --init training/core
else:
    %cd /content/HollyShit
    !git pull
    !git submodule update --init training/core

# 再次硬校验挂载点后，创建真实 Google Drive Checkpoint、数据与向量存储目录并建立软链接
assert os.path.ismount('/content/drive'), '❌ /content/drive 未挂载！严禁在本地容器盘创建伪目录！'
os.makedirs('/content/drive/MyDrive/hollyshit/checkpoints/p0_v3', exist_ok=True)
os.makedirs('/content/drive/MyDrive/hollyshit/data', exist_ok=True)
os.makedirs('/content/drive/MyDrive/hollyshit/embs', exist_ok=True)
!ln -sfn /content/drive/MyDrive/hollyshit/checkpoints checkpoints
!ln -sfn /content/drive/MyDrive/hollyshit/data data

# 数据源配置：按需设置为 'ms' (默认魔搭直连) / 'gdrive' (云盘离线源) / 'hf' (Hugging Face) / 'local'
DATA_SOURCE = "ms"
GDRIVE_DIR = "/content/drive/MyDrive/hollyshit/embs"
os.environ['DATA_SOURCE'] = DATA_SOURCE
os.environ['GDRIVE_DIR'] = GDRIVE_DIR

print('✅ 仓库与真实 Google Drive 云盘软链接就绪:', os.getcwd())
print(f'⚙️ 当前激活数据源: DATA_SOURCE={DATA_SOURCE!r}, GDRIVE_DIR={GDRIVE_DIR!r}')


In [ ]:
import json
import os
import re
import torch
from huggingface_hub import HfApi

# 1. 确保工作目录在 /content/HollyShit，并创建本地高速 NVMe SSD 分块缓存目录 (严禁放在 Google Drive!)
if os.path.isdir('/content/HollyShit'):
    os.chdir('/content/HollyShit')
local_embs_cache = '/content/embs_cache'
os.makedirs(local_embs_cache, exist_ok=True)

# 2. 加载或自愈生成 dataset_config.json
if not os.path.exists('dataset_config.json'):
    default_ds_cfg = {
        'default_source': 'ms',
        'modelscope_repo_id': 'catclever/emb_npy',
        'huggingface_repo_id': 'drunkencat/emb_npy',
        'gdrive_dir': '/content/drive/MyDrive/hollyshit/embs',
        'base_models': ['bge', 'qwen', 'xiaobu', 'youtu', 'conan_v1'],
        'chunk_name_patterns': {
            'bge': 'bge_m3/bge_embeddings_chunk_{start:07d}_{end:07d}.npz',
            'qwen': 'gte_qwen/gte_qwen2_embeddings_chunk_{start:07d}_{end:07d}.npz',
            'xiaobu': 'xiaobu/xiaobu_embeddings_chunk_{start:07d}_{end:07d}.npz',
            'youtu': 'youtu/youtu_embeddings_chunk_{start:07d}_{end:07d}.npz',
            'conan_v1': 'conan/conan_v1_embeddings_chunk_{start:07d}_{end:07d}.npz',
        },
        'emb_dims': [1024, 3584, 1792, 2048, 1792],
        'parquet_path': 'data/Basic_ZH/chunked_mixed_omni.parquet',
    }
    with open('dataset_config.json', 'w', encoding='utf-8') as _wf:
        json.dump(default_ds_cfg, _wf, indent=4, ensure_ascii=False)
    print('🛠️ 已自动生成默认 dataset_config.json')

with open('dataset_config.json', 'r', encoding='utf-8') as f:
    ds_cfg = json.load(f)

models = ds_cfg['base_models']
patterns = ds_cfg['chunk_name_patterns']
emb_dims = ds_cfg['emb_dims']
cand_hf_repo = os.environ.get('HF_DATASET_REPO') or ds_cfg.get('huggingface_repo_id', 'drunkencat/emb_npy')
ms_repo_id = ds_cfg.get('modelscope_repo_id', 'catclever/emb_npy')
active_data_source = os.environ.get('DATA_SOURCE', ds_cfg.get('default_source', 'ms'))
active_gdrive_dir = os.environ.get('GDRIVE_DIR', ds_cfg.get('gdrive_dir', '/content/drive/MyDrive/hollyshit/embs'))
ALLOW_MS_FALLBACK = True

def _count_aligned_chunks(file_list):
    per_m = {}
    for m_name in models:
        prefix = patterns[m_name].split('/')[0]
        bounds = {}
        for rf in file_list:
            rf_norm = rf.replace('\\', '/')
            if rf_norm.endswith('.npz') and (rf_norm.startswith(prefix + '/') or f'/{prefix}/' in rf_norm):
                mm = re.search(r'chunk_(\d+)_(\d+)\.npz', os.path.basename(rf_norm))
                if mm:
                    bounds[int(mm.group(1))] = int(mm.group(2))
        per_m[m_name] = bounds
    common = sorted(set.intersection(*[set(b.keys()) for b in per_m.values()])) if per_m else []
    clauses = min(per_m[m][common[-1]] for m in models) if common else 0
    return per_m, common, clauses

# 3. 按照选定的 active_data_source 校验分块可用性
print(f'📡 当前选择的数据源: {active_data_source.upper()}')
if active_data_source == 'gdrive':
    assert os.path.ismount('/content/drive'), '❌ 使用 gdrive 源必须首先挂载 /content/drive！'
    import glob
    gdrive_files = glob.glob(os.path.join(active_gdrive_dir, '**', '*.npz'), recursive=True)
    gd_per_model, gd_common, gd_clauses = _count_aligned_chunks(gdrive_files)
    print(f'📂 [GDrive] 目录: {active_gdrive_dir}，探测到 {len(gd_common)} 个对齐宏分块 ({gd_clauses:,} 条子句)')
    os.environ['ACTIVE_HF_REPO'] = ''
elif active_data_source == 'hf':
    try:
        hf_files = list(HfApi(token=os.environ.get('HF_TOKEN') or None).list_repo_files(repo_id=cand_hf_repo, repo_type='dataset'))
        hf_per_model, hf_common, hf_clauses = _count_aligned_chunks(hf_files)
        os.environ['ACTIVE_HF_REPO'] = cand_hf_repo
        print(f'✅ Hugging Face 源 [{cand_hf_repo}]: 共 {len(hf_common)} 个对齐宏分块，总计 {hf_clauses:,} 条子句。')
    except Exception as _e:
        print(f'⚠️ Hugging Face 源不可用: {_e}，回退到 ModelScope')
        active_data_source = 'ms'
        os.environ['DATA_SOURCE'] = 'ms'
        os.environ['ACTIVE_HF_REPO'] = ''

if active_data_source == 'ms':
    from modelscope.hub.api import HubApi
    ms_api = HubApi()
    if os.environ.get('MODELSCOPE_API_TOKEN'):
        try:
            ms_api.login(os.environ['MODELSCOPE_API_TOKEN'])
        except Exception as e:
            print(f'⚠️ ModelScope Token 鉴权跳过: {e}')
    ms_files = [item.get('Path', item.get('Name', '')) for item in ms_api.get_dataset_files(ms_repo_id, recursive=True)]
    ms_per_model, ms_common, ms_clauses = _count_aligned_chunks(ms_files)
    assert len(ms_common) > 0, '未在魔搭探测到分块，请检查网络！'
    os.environ['ACTIVE_HF_REPO'] = ''
    print(f'✅ 已启用 ModelScope 默认流式源 [{ms_repo_id}]！共 {len(ms_common)} 个对齐宏分块，总计 {ms_clauses:,} 条子句。')

# 4. 按 GPU 档位 (L4/A100 vs T4) 自适应设定最优超参并注入环境变量
vram_gb = (torch.cuda.get_device_properties(0).total_memory / (1024.0 ** 3)) if torch.cuda.is_available() else 0.0
ram_gb = 0.0
if os.path.exists('/proc/meminfo'):
    with open('/proc/meminfo', 'r', encoding='utf-8') as f:
        for line in f:
            if line.startswith('MemTotal:'):
                ram_gb = int(line.split()[1]) / (1024.0 ** 2)
                break

if vram_gb >= 22.0 and ram_gb >= 24.0:
    p0_cfg = {
        'P0_BATCH_SIZE': '512',
        'P0_TRIPLET_BATCH': '128',
        'P0_LR': '4e-4',
        'P0_WARMUP': '1000',
        'P0_TOTAL_STEPS': '50750',
    }
    tier_label = 'L4 / A100 高吞吐大批量档 (Batch=512)'
else:
    p0_cfg = {
        'P0_BATCH_SIZE': '256',
        'P0_TRIPLET_BATCH': '64',
        'P0_LR': '3e-4',
        'P0_WARMUP': '2000',
        'P0_TOTAL_STEPS': '101500',
    }
    tier_label = 'T4 稳健标准档 (Batch=256)'

os.environ.update(p0_cfg)
os.environ.setdefault('HF_CKPT_REPO', '')
print(f'⚙️ 已激活训练配置 [{tier_label}]: {p0_cfg} | DATA_SOURCE={os.environ.get("DATA_SOURCE")!r} | ACTIVE_HF_REPO={os.environ.get("ACTIVE_HF_REPO")!r}')


In [ ]:
# ==============================================================================
# （可选）直接下载分块至 Google Drive（存好后将 DATA_SOURCE 设为 'gdrive' 即可完全离线/零外部网络依赖极速训练）
# - 使用 ModelScope dataset_file_download 直连极速下载到 Google Drive 持久目录
# - 一旦云盘集齐分块，后续训练时只需将 DATA_SOURCE = 'gdrive'，训练进程将全自动由后台打工仔
#   将正在消费的分块预取暂存 (Stage) 至本地高速 NVMe SSD (/content/embs_cache)，
#   消费完毕自动清除 NVMe 暂存，对 Google Drive 原始分块 100% 零修改、零误删、极速训练！
# ==============================================================================
DOWNLOAD_TO_GDRIVE = False  # 若需要将全部或部分分块预先保存在 Google Drive，请改为 True

if DOWNLOAD_TO_GDRIVE:
    assert os.path.ismount('/content/drive'), '❌ /content/drive 未挂载！'
    import json
    import os
    from modelscope.hub.file_download import dataset_file_download
    from modelscope.hub.api import HubApi

    os.makedirs(GDRIVE_DIR, exist_ok=True)
    with open('dataset_config.json', 'r', encoding='utf-8') as f:
        _cfg = json.load(f)

    _models = _cfg['base_models']
    _patterns = _cfg['chunk_name_patterns']
    _ms_repo = _cfg.get('modelscope_repo_id', 'catclever/emb_npy')

    print(f'📥 开始同步 ModelScope ({_ms_repo}) 分块至 Google Drive: {GDRIVE_DIR} ...')
    _api = HubApi()
    if os.environ.get('MODELSCOPE_API_TOKEN'):
        try:
            _api.login(os.environ['MODELSCOPE_API_TOKEN'])
        except Exception:
            pass
    _files = [item.get('Path', item.get('Name', '')) for item in _api.get_dataset_files(_ms_repo, recursive=True)]
    _npz_files = [f for f in _files if f.endswith('.npz')]
    print(f'📦 远端共检测到 {len(_npz_files)} 个 .npz 分块文件。')

    for idx, remote_fn in enumerate(_npz_files, 1):
        target_dest = os.path.join(GDRIVE_DIR, remote_fn)
        if os.path.exists(target_dest) and os.path.getsize(target_dest) > 1024:
            continue
        print(f'  [{idx}/{len(_npz_files)}] 正在下载: {remote_fn} -> {GDRIVE_DIR} ...')
        downloaded = dataset_file_download(_ms_repo, remote_fn, cache_dir=GDRIVE_DIR)
        print(f'  ✅ 已保存: {downloaded}')
    print('🎉 Google Drive 离线分块同步完毕！请将 DATA_SOURCE 设为 "gdrive" 开启完全离线训练！')


In [ ]:
# ==============================================================================
# 🚀 启动 Phase 0 v3: 去空腔各向同性五师神之流形训练
# - 启动前双重硬门禁校验 os.path.ismount('/content/drive')，杜绝未挂载静默写临时盘
# - --data_source $DATA_SOURCE & --gdrive_dir "$GDRIVE_DIR": 统一多数据源 (ms / gdrive / hf / local)
# - --hf_repo_id "$ACTIVE_HF_REPO" & --hf_ckpt_repo "$HF_CKPT_REPO": HF CDN 流式 + 异地备份
# - --no-prefetch & --keep_last_ckpts 3 & --auto_resume: 单块控流 + 自动断点续训
# ==============================================================================
import os
assert os.path.ismount('/content/drive'), '❌ /content/drive 未挂载！已紧急拦截训练启动，请先挂载 Google Drive！'

%cd /content/HollyShit
!python -u training/train_phase0_v3_cuda.py \
  --config_file dataset_config.json \
  --data_source $DATA_SOURCE \
  --gdrive_dir "$GDRIVE_DIR" \
  --hf_repo_id "$ACTIVE_HF_REPO" \
  --hf_ckpt_repo "$HF_CKPT_REPO" \
  --data_dir /content/embs_cache \
  --download_cache_dir /content/embs_cache \
  --no-prefetch \
  --batch_size $P0_BATCH_SIZE \
  --triplet_batch_size $P0_TRIPLET_BATCH \
  --epochs 3 \
  --lr $P0_LR \
  --warmup_steps $P0_WARMUP \
  --total_steps $P0_TOTAL_STEPS \
  --out_dir /content/drive/MyDrive/hollyshit/checkpoints/p0_v3 \
  --ckpt_prefix p0_v3_5models \
  --eval_steps 200 \
  --save_steps 5000 \
  --keep_last_ckpts 3 \
  --auto_resume


In [ ]:
import os
import shutil
import sys
import time
import torch
from google.colab import drive

# 1. 收尾前首先硬校验 /content/drive 确实处于挂载状态
assert os.path.ismount('/content/drive'), '❌ /content/drive 未挂载！无法校验或安全刷写云盘检查点！'

if '/content/HollyShit' not in sys.path:
    sys.path.insert(0, '/content/HollyShit')
from training.train_phase0_v3_cuda import (
    mirror_checkpoint_to_hf_hub,
    resolve_latest_phase0_v3_checkpoint,
)

drive_ckpt_root = '/content/drive/MyDrive/hollyshit/checkpoints/p0_v3'
ckpt_prefix = 'p0_v3_5models'
latest_ckpt = resolve_latest_phase0_v3_checkpoint(drive_ckpt_root, ckpt_prefix)
assert latest_ckpt is not None, f'❌ 未在云盘目录 {drive_ckpt_root} 中检测到任何有效检查点！'

print('=' * 68)
print(f'🏆 最新有效 Phase 0 v3 检查点: {latest_ckpt}')
ms_path = os.path.join(latest_ckpt, 'manifold_stats.pt')
if os.path.isfile(ms_path):
    ms = torch.load(ms_path, map_location='cpu')
    hm = ms.get('health_metrics', {})
    print(f"   • global_step : {ms.get('global_step')}")
    print(f"   • arch        : emb_dims={ms.get('emb_dims')} -> d_model={ms.get('d_model')} -> z_dim={ms.get('z_dim')} (sphere={ms.get('sphere_scale')})")
    if hm:
        print(
            f"   • health      : PR={hm.get('participation_ratio', 0.0):.1f} | "
            f"ReconCos={hm.get('mean_teacher_recon_cos', 0.0):.4f} | "
            f"RelSpearman={hm.get('relational_spearman', 0.0):.4f} | "
            f"MidpointCos={hm.get('midpoint_cavity_recon_cos', 0.0):.4f} | "
            f"CloudTop1={hm.get('cloud_view_top1_acc', 0.0):.4f} | "
            f"NextTop1={hm.get('next_clause_top1_acc', 0.0):.4f} | "
            f"AdjAUC={hm.get('adjacent_auc', 0.0):.4f}"
        )

# 2. 完整同步全部 9 项训练与下游蒸馏产物至 p0_v3_5models_best
all_export_artifacts = [
    'sense_fuser.safetensors',
    'god_encoder.safetensors',
    'sense_fuser.pt',
    'god_encoder.pt',
    'recon_heads.pt',
    'next_clause_head.pt',
    'manifold_stats.pt',
    'training_args.json',
    'dataloader.json',
]
best_alias_dir = os.path.join(drive_ckpt_root, f'{ckpt_prefix}_best')
os.makedirs(best_alias_dir, exist_ok=True)
for fn in all_export_artifacts:
    src_file = os.path.join(latest_ckpt, fn)
    if os.path.isfile(src_file):
        shutil.copy2(src_file, os.path.join(best_alias_dir, fn))

# 硬校验核心落盘文件存在且字节数 > 0
for must_fn in ('god_encoder.safetensors', 'sense_fuser.safetensors', 'manifold_stats.pt'):
    must_path = os.path.join(best_alias_dir, must_fn)
    assert os.path.isfile(must_path) and os.path.getsize(must_path) > 0, (
        f'❌ 关键权重文件缺失或为空: {must_path}'
    )
print(f'🔗 已完整同步 p0_v3_5models_best (共 {len(os.listdir(best_alias_dir))} 个文件): {best_alias_dir}')

# 若配置了 HF_TOKEN / HF_CKPT_REPO，同步将 best 目录异地备份至 Hugging Face 私有模型库
hf_ckpt_repo = os.environ.get('HF_CKPT_REPO', '').strip()
if hf_ckpt_repo:
    mirror_checkpoint_to_hf_hub(best_alias_dir, hf_ckpt_repo)
print('=' * 68)

# 3. 切回 /content 释放软链接目录句柄后，强制刷写并卸载 Google Drive FUSE 缓存 (严禁吞掉异常!)
os.chdir('/content')
print('🔄 已切回 /content，正在执行 drive.flush_and_unmount() 强制刷盘...')
flush_ok = False
drive.flush_and_unmount()
flush_ok = not os.path.ismount('/content/drive')
assert flush_ok, '❌ drive.flush_and_unmount() 后 /content/drive 仍处于挂载态，请勿销毁实例！'
print('✅ Google Drive FUSE 写入缓存已 100% 安全落盘并完成卸载！')

# ⚠️ 安全铁律：默认禁用 runtime.unassign()！请先在 Google Drive 网页端确认 p0_v3_5models_best 权重已存在。
# 如需开启无人值守自动释放实例，必须保留 `if flush_ok:` 硬门禁（严禁在 flush 失败时销毁虚拟机）：
# if flush_ok:
#     from google.colab import runtime
#     time.sleep(5)
#     runtime.unassign()
